# v4 — Inference-Only Notebook (Per-Image Normalization)

**Model**: `DualSwinFusionSeg` — Dual Swin V2 Small, UNet++ decoder, concat1x1 fusion

**Key change vs v3**: Per-image percentile normalization (domain-invariant).
Each band is clipped to its own image's P1/P99 → [0,1], then z-scored with global mean/std.
This eliminates the catastrophic DEM domain shift between archive training data and phase2 test data.

**Usage**:
1. Set `TEST_DATA_DIR` to the test image folder (raw `.tif`)
2. Set `CHECKPOINT_DIR` to folder containing `fold{1-5}_best.pt`
3. Set `STATS_JSON_PATH` to the `norm_stats_v4.json` produced by v4 training
4. Run all cells → outputs a `submission.zip` of predicted mask TIFs

In [24]:
# ============================================================
# Cell 1 — Imports & device & determinism
# ============================================================
import os, json, math, zipfile, warnings, random
from pathlib import Path

import numpy as np
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
import rasterio
import timm

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name()}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'  cudnn.deterministic = {torch.backends.cudnn.deterministic}')

Device: cuda
  GPU: NVIDIA GeForce RTX 4090
  VRAM: 25.2 GB
  cudnn.deterministic = True


In [25]:
# ============================================================
# Cell 2 — ★ CONFIGURABLE PATHS ★
# ============================================================
import os
from pathlib import Path

# Ensure CWD is always the project root (works from notebooks/ or root)
_PROJECT_ROOT = Path.cwd()
if _PROJECT_ROOT.name == "notebooks":
    _PROJECT_ROOT = _PROJECT_ROOT.parent
os.chdir(_PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

# --- TEST DATA (raw .tif files) ---
# Phase 1 test set:
# TEST_DATA_DIR = "data/phase1_dataset/test/images"
# Phase 2 test set:
TEST_DATA_DIR = "data/phase2_dataset/test/images"  # ★ CHANGE

# --- CHECKPOINTS (folder containing fold1_best.pt … fold5_best.pt) ---
CHECKPOINT_DIR = "output/nb_training_output/checkpoints/swinv2_unetplusplus_concat1x1"  # ★ CHANGE

# --- OUTPUT ---
OUTPUT_DIR = "output/nb_inference_output"
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- NORMALIZATION STATS ---
STATS_JSON_PATH = "output/nb_training_output/kfold_results_v4/norm_stats_v4.json"  # ★ CHANGE

# If STATS_JSON_PATH is None, recompute stats from this folder of training+val TIFs:
COMPUTE_STATS_FROM = "data/phase1_dataset"  # only used when STATS_JSON_PATH is None

# --- MODEL / INFERENCE CONFIG ---
ENCODER_NAME  = "swinv2_small_window8_256"
DECODER_NAME  = "unetplusplus"
FUSION_NAME   = "concat1x1"
IMG_SIZE      = 128    # v4 trains at native 128
ORIG_SIZE     = 128    # submission masks at this resolution
FPN_CHANNELS  = 256
BATCH_SIZE    = 8
NUM_WORKERS   = 0      # must be 0 in notebooks (multiprocessing can't pickle notebook classes)
THRESH        = 0.51
USE_TTA       = True   # 4-fold TTA (original + h-flip + v-flip + hv-flip)
NUM_FOLDS     = 5

# ============================================================
# Derived — DO NOT EDIT
# ============================================================
BAND_INDICES   = [5, 6, 7, 3, 2, 1, 4]
RGB_BANDS      = [5, 6, 7]
AUX_BANDS      = [3, 2, 1, 4]
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

import numpy as np
RGB_STAT_IDX = np.array([BAND_INDICES.index(b) for b in RGB_BANDS])
AUX_STAT_IDX = np.array([BAND_INDICES.index(b) for b in AUX_BANDS])

print("Configuration")
print("=" * 60)
print(f"Test data:   {TEST_DATA_DIR}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Output:      {OUTPUT_DIR}")
print(f"Stats JSON:  {STATS_JSON_PATH}")
print(f"Device:      {DEVICE}")
print(f"TTA:         {'ON' if USE_TTA else 'OFF'}")
print(f"Threshold:   {THRESH}")
print(f"Folds:       {NUM_FOLDS}")
print(f"Image size:  {IMG_SIZE}")
print("=" * 60)


Working directory: /home/user/MLS/Mars-LS-Segmentation
Configuration
Test data:   data/phase2_dataset/test/images
Checkpoints: output/nb_training_output/checkpoints/swinv2_unetplusplus_concat1x1
Output:      output/nb_inference_output
Stats JSON:  output/nb_training_output/kfold_results_v4/norm_stats_v4.json
Device:      cuda
TTA:         ON
Threshold:   0.51
Folds:       5
Image size:  128


In [26]:
# ============================================================
# Cell 3 — ★ v4: Per-Image Normalization + Load Stats from JSON
# ============================================================

def normalize_bands_per_image(arr_chw, p_low=1.0, p_high=99.0):
    """Per-image, per-band percentile normalization — domain-invariant.

    Each band is clipped to its OWN image's [P_low, P_high] percentiles and
    rescaled to [0, 1].  This eliminates sensitivity to absolute value shifts
    between domains (e.g. DEM at different elevations).
    """
    C = arr_chw.shape[0]
    out = np.empty_like(arr_chw, dtype=np.float32)
    for c in range(C):
        flat = arr_chw[c].ravel()
        lo = np.percentile(flat, p_low)
        hi = np.percentile(flat, p_high)
        hi = max(hi, lo + 1e-6)
        out[c] = np.clip((arr_chw[c].astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    return out


def compute_mean_std_per_image_norm(img_paths, band_indices,
                                     p_low=1.0, p_high=99.0,
                                     max_files=None, max_pixels=2_000_000):
    """Compute channel mean/std AFTER per-image percentile normalization."""
    paths = list(img_paths)
    if max_files is not None:
        paths = paths[:max_files]
    C = len(band_indices)
    sums = np.zeros(C, dtype=np.float64)
    sqs  = np.zeros(C, dtype=np.float64)
    n    = 0
    rng  = np.random.default_rng(123)
    for p in tqdm(paths, desc="Compute mean/std (per-image norm)"):
        with rasterio.open(str(p)) as src:
            arr = src.read(band_indices).astype(np.float32)
        arr = normalize_bands_per_image(arr, p_low, p_high)
        flat = arr.reshape(C, -1)
        if flat.shape[1] > 20000:
            idx = rng.choice(flat.shape[1], size=20000, replace=False)
            flat = flat[:, idx]
        sums += flat.sum(axis=1)
        sqs  += (flat * flat).sum(axis=1)
        n    += flat.shape[1]
        if n >= max_pixels:
            break
    means = (sums / max(n, 1)).astype(np.float32)
    vars_ = (sqs  / max(n, 1) - means.astype(np.float64) ** 2).clip(min=1e-8)
    stds  = np.sqrt(vars_).astype(np.float32)
    return means, stds


# ------- Load stats from JSON or recompute -------
if STATS_JSON_PATH and Path(STATS_JSON_PATH).exists():
    with open(STATS_JSON_PATH) as f:
        _s = json.load(f)
    MEANS = np.array(_s['channel_means'], dtype=np.float32)
    STDS  = np.array(_s['channel_stds'],  dtype=np.float32)
    print(f'Loaded per-image norm stats from {STATS_JSON_PATH}')
    print(f'  normalization: {_s.get("normalization", "unknown")}')
    print(f'  p_low={_s.get("p_low", "?")}, p_high={_s.get("p_high", "?")}')
else:
    print(f'STATS_JSON_PATH not found — recomputing from {COMPUTE_STATS_FROM}')
    _root = Path(COMPUTE_STATS_FROM)
    _all_imgs = sorted(
        list((_root / 'train' / 'images').glob('*.tif')) +
        list((_root / 'val'   / 'images').glob('*.tif'))
    )
    print(f'  Found {len(_all_imgs)} train+val images for stats computation')
    assert len(_all_imgs) > 0, f'No .tif images found!'
    MEANS, STDS = compute_mean_std_per_image_norm(
        [str(p) for p in _all_imgs], BAND_INDICES)
    # Save for next time
    _save = {
        'normalization': 'per_image_percentile',
        'p_low': 1.0, 'p_high': 99.0,
        'band_indices': BAND_INDICES, 'rgb_bands': RGB_BANDS, 'aux_bands': AUX_BANDS,
        'channel_means': MEANS.tolist(), 'channel_stds': STDS.tolist(),
        'img_size': IMG_SIZE,
    }
    _out_stats = OUTPUT_DIR / 'norm_stats_v4.json'
    _out_stats.write_text(json.dumps(_save, indent=2))
    print(f'  Stats saved → {_out_stats}')

print(f'\n★ v4 Per-Image Normalization Stats ({len(BAND_INDICES)} channels):')
print(f'  mean = {MEANS}')
print(f'  std  = {STDS}')
print(f'  (No global low/high — each test image uses its own P1/P99)')

STATS_JSON_PATH not found — recomputing from data/phase1_dataset
  Found 531 train+val images for stats computation


Compute mean/std (per-image norm):   4%|▍         | 21/531 [00:00<00:04, 104.19it/s]

Compute mean/std (per-image norm):  23%|██▎       | 122/531 [00:01<00:03, 105.61it/s]

  Stats saved → output/nb_inference_output/norm_stats_v4.json

★ v4 Per-Image Normalization Stats (7 channels):
  mean = [0.42286202 0.3531582  0.311243   0.5828425  0.31569043 0.39825553
 0.51703095]
  std  = [0.26482505 0.24576192 0.2488724  0.33187866 0.2858247  0.2262664
 0.20617135]
  (No global low/high — each test image uses its own P1/P99)


In [27]:
# ============================================================
# Cell 4 — Model Architecture (copied from v3 training notebook)
# ============================================================

# --- Backbone helpers ---
def adapt_patch_embed_in_chans(model, in_chans_new):
    pe = model.patch_embed
    old_conv = pe.proj
    old_w = old_conv.weight.data
    embed_dim, old_in, kh, kw = old_w.shape
    new_conv = nn.Conv2d(
        in_chans_new, embed_dim,
        kernel_size=old_conv.kernel_size, stride=old_conv.stride,
        padding=old_conv.padding, bias=(old_conv.bias is not None),
    )
    with torch.no_grad():
        new_w = torch.zeros(embed_dim, in_chans_new, kh, kw, device=old_w.device)
        new_w[:, :3, :, :] = old_w
        if in_chans_new > 3:
            rgb_mean = old_w.mean(dim=1, keepdim=True)
            new_w[:, 3:, :, :] = rgb_mean.repeat(1, in_chans_new - 3, 1, 1)
        new_conv.weight.copy_(new_w)
        if old_conv.bias is not None:
            new_conv.bias.copy_(old_conv.bias.data)
    pe.proj = new_conv
    return model

def make_swin_features(encoder_name, pretrained=True, img_size=128):
    enc = timm.create_model(
        encoder_name, pretrained=pretrained,
        features_only=True, out_indices=(0, 1, 2, 3), img_size=img_size,
    )
    if hasattr(enc, 'patch_embed'):
        enc.patch_embed.img_size = None
        if hasattr(enc.patch_embed, 'strict_img_size'):
            enc.patch_embed.strict_img_size = False
    return enc

def to_nchw(feats, in_chs):
    out = []
    for f, c in zip(feats, in_chs):
        if f.ndim == 4 and f.shape[-1] == c and f.shape[1] != c:
            f = f.permute(0, 3, 1, 2).contiguous()
        out.append(f)
    return out

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

# --- UNet++ Decoder (used in v3) ---
class UNetPlusPlusDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        C = fpn_channels
        self.reduce = nn.ModuleList([ConvBNReLU(c, C, k=1, s=1, p=0) for c in in_channels_list])
        def _node(n_in): return nn.Sequential(ConvBNReLU(C * n_in, C), ConvBNReLU(C, C))
        self.x01 = _node(2); self.x11 = _node(2); self.x21 = _node(2)
        self.x02 = _node(3); self.x12 = _node(3)
        self.x03 = _node(4)
        self.final = nn.Sequential(ConvBNReLU(C, C), nn.Dropout2d(0.1))
    @staticmethod
    def _up(x, t): return F.interpolate(x, size=t.shape[-2:], mode='bilinear', align_corners=False)
    def forward(self, feats):
        x00, x10, x20, x30 = [self.reduce[i](f) for i, f in enumerate(feats)]
        x21 = self.x21(torch.cat([x20, self._up(x30, x20)], 1))
        x11 = self.x11(torch.cat([x10, self._up(x20, x10)], 1))
        x01 = self.x01(torch.cat([x00, self._up(x10, x00)], 1))
        x12 = self.x12(torch.cat([x10, x11, self._up(x21, x10)], 1))
        x02 = self.x02(torch.cat([x00, x01, self._up(x11, x00)], 1))
        x03 = self.x03(torch.cat([x00, x01, x02, self._up(x12, x00)], 1))
        return self.final(x03)

# --- Other decoders (kept for flexibility) ---
class PPM(nn.Module):
    def __init__(self, in_ch, out_ch, pool_sizes=(1, 2, 3, 6)):
        super().__init__()
        inter = max(out_ch // len(pool_sizes), 32)
        self.stages = nn.ModuleList([nn.Sequential(
            nn.AdaptiveAvgPool2d(ps), nn.Conv2d(in_ch, inter, 1, bias=False),
            nn.BatchNorm2d(inter), nn.ReLU(inplace=True)) for ps in pool_sizes])
        self.bottleneck = nn.Sequential(
            nn.Conv2d(in_ch + inter * len(pool_sizes), out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        h, w = x.shape[-2:]
        outs = [x] + [F.interpolate(s(x), size=(h, w), mode='bilinear', align_corners=False) for s in self.stages]
        return self.bottleneck(torch.cat(outs, 1))

class UPerNetDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        self.ppm = PPM(in_channels_list[-1], fpn_channels)
        self.lateral_convs = nn.ModuleList([nn.Conv2d(c, fpn_channels, 1) for c in in_channels_list[:-1]])
        self.fpn_convs = nn.ModuleList([ConvBNReLU(fpn_channels, fpn_channels) for _ in in_channels_list[:-1]])
        self.fuse = nn.Sequential(ConvBNReLU(fpn_channels * 4, fpn_channels), nn.Dropout2d(0.1))
    def forward(self, feats):
        c1, c2, c3, c4 = feats
        p4 = self.ppm(c4)
        p3 = self.lateral_convs[2](c3) + F.interpolate(p4, size=c3.shape[-2:], mode='bilinear', align_corners=False)
        p2 = self.lateral_convs[1](c2) + F.interpolate(p3, size=c2.shape[-2:], mode='bilinear', align_corners=False)
        p1 = self.lateral_convs[0](c1) + F.interpolate(p2, size=c1.shape[-2:], mode='bilinear', align_corners=False)
        p3 = self.fpn_convs[2](p3); p2 = self.fpn_convs[1](p2); p1 = self.fpn_convs[0](p1)
        h, w = p1.shape[-2:]
        return self.fuse(torch.cat([p1,
            F.interpolate(p2, (h,w), mode='bilinear', align_corners=False),
            F.interpolate(p3, (h,w), mode='bilinear', align_corners=False),
            F.interpolate(p4, (h,w), mode='bilinear', align_corners=False)], 1))

class SegFormerMLPDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        self.linear_projs = nn.ModuleList([nn.Sequential(
            nn.Conv2d(c, fpn_channels, 1, bias=False), nn.BatchNorm2d(fpn_channels), nn.ReLU(inplace=True))
            for c in in_channels_list])
        self.fuse = nn.Sequential(
            nn.Conv2d(fpn_channels * 4, fpn_channels, 1, bias=False),
            nn.BatchNorm2d(fpn_channels), nn.ReLU(inplace=True), nn.Dropout2d(0.1))
    def forward(self, feats):
        t = feats[0].shape[-2:]
        outs = [F.interpolate(self.linear_projs[i](f), size=t, mode='bilinear', align_corners=False) for i, f in enumerate(feats)]
        return self.fuse(torch.cat(outs, 1))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=(6, 12, 18)):
        super().__init__()
        modules = [nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))]
        for r in rates:
            modules.append(nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)))
        modules.append(nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)))
        self.convs = nn.ModuleList(modules)
        self.project = nn.Sequential(nn.Conv2d(out_ch * (2 + len(rates)), out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        h, w = x.shape[-2:]
        outs = [F.interpolate(c(x), size=(h,w), mode='bilinear', align_corners=False) if c(x).shape[-2:] != (h,w) else c(x) for c in self.convs]
        return self.project(torch.cat(outs, 1))

class DeepLabV3PlusDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        self.aspp = ASPP(in_channels_list[-1], fpn_channels)
        self.low_proj = nn.Sequential(nn.Conv2d(in_channels_list[0], 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(ConvBNReLU(fpn_channels + 48, fpn_channels), ConvBNReLU(fpn_channels, fpn_channels), nn.Dropout2d(0.1))
    def forward(self, feats):
        c1, _, _, c4 = feats
        x = F.interpolate(self.aspp(c4), size=c1.shape[-2:], mode='bilinear', align_corners=False)
        return self.fuse(torch.cat([x, self.low_proj(c1)], 1))

class SimpleFPNDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        self.lat = nn.ModuleList([nn.Conv2d(c, fpn_channels, 1) for c in in_channels_list])
        self.out = nn.ModuleList([ConvBNReLU(fpn_channels, fpn_channels) for _ in in_channels_list])
        self.fuse = nn.Sequential(ConvBNReLU(fpn_channels * 4, fpn_channels), nn.Dropout2d(0.1))
    def forward(self, feats):
        lat = [self.lat[i](f) for i, f in enumerate(feats)]
        for i in range(len(lat)-1, 0, -1):
            lat[i-1] = lat[i-1] + F.interpolate(lat[i], size=lat[i-1].shape[-2:], mode='bilinear', align_corners=False)
        outs = [self.out[i](lat[i]) for i in range(len(lat))]
        t = outs[0].shape[-2:]
        return self.fuse(torch.cat([F.interpolate(o, size=t, mode='bilinear', align_corners=False) for o in outs], 1))

def build_decoder(name, in_channels_list, fpn_channels=256):
    name = name.lower()
    if name == 'upernet':       return UPerNetDecoder(in_channels_list, fpn_channels)
    if name == 'segformer_mlp': return SegFormerMLPDecoder(in_channels_list, fpn_channels)
    if name == 'deeplabv3plus': return DeepLabV3PlusDecoder(in_channels_list, fpn_channels)
    if name == 'unetplusplus':  return UNetPlusPlusDecoder(in_channels_list, fpn_channels)
    if name == 'fpn':           return SimpleFPNDecoder(in_channels_list, fpn_channels)
    raise ValueError(f'Unknown decoder: {name}')

# --- Fusion strategies ---
class FusionLateLogits(nn.Module):
    def forward(self, A, B): return A, B   # handled specially in model.forward

class FusionConcat1x1(nn.Module):
    def __init__(self, chs):
        super().__init__()
        self.proj = nn.ModuleList([nn.Conv2d(2*c, c, 1) for c in chs])
    def forward(self, A, B):
        return [self.proj[i](torch.cat([a, b], 1)) for i, (a, b) in enumerate(zip(A, B))]

class FusionWeightedSum(nn.Module):
    def __init__(self, chs):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(len(chs)))
        self.beta  = nn.Parameter(torch.ones(len(chs)))
    def forward(self, A, B):
        return [self.alpha[i]*a + self.beta[i]*b for i, (a, b) in enumerate(zip(A, B))]

class FusionGated(nn.Module):
    def __init__(self, chs, r=16):
        super().__init__()
        self.gates = nn.ModuleList()
        for c in chs:
            mid = max(c//r, 8)
            self.gates.append(nn.Sequential(
                nn.AdaptiveAvgPool2d(1), nn.Conv2d(2*c, mid, 1), nn.ReLU(inplace=True),
                nn.Conv2d(mid, c, 1), nn.Sigmoid()))
    def forward(self, A, B):
        return [g(torch.cat([a,b],1))*a + (1-g(torch.cat([a,b],1)))*b for g,a,b in zip(self.gates,A,B)]

class FusionFiLM(nn.Module):
    def __init__(self, chs, r=16):
        super().__init__()
        self.film = nn.ModuleList()
        for c in chs:
            mid = max(c//r, 8)
            self.film.append(nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, mid, 1), nn.ReLU(inplace=True), nn.Conv2d(mid, 2*c, 1)))
    def forward(self, A, B):
        out = []
        for i, (a, b) in enumerate(zip(A, B)):
            gb = self.film[i](b)
            gamma, beta = torch.chunk(gb, 2, 1)
            out.append((1+gamma)*a + beta)
        return out

class FusionCrossAttention(nn.Module):
    def __init__(self, chs, num_heads=4):
        super().__init__()
        self.Qs = nn.ModuleList([nn.Conv2d(c,c,1) for c in chs])
        self.Ks = nn.ModuleList([nn.Conv2d(c,c,1) for c in chs])
        self.Vs = nn.ModuleList([nn.Conv2d(c,c,1) for c in chs])
        self.attn = nn.ModuleList([nn.MultiheadAttention(c, num_heads=num_heads, batch_first=True) for c in chs])
        self.out  = nn.ModuleList([nn.Conv2d(c,c,1) for c in chs])
    def forward(self, A, B):
        outs = []
        for i, (a, b) in enumerate(zip(A, B)):
            Bn, C, H, W = a.shape
            q = self.Qs[i](a).flatten(2).transpose(1,2)
            k = self.Ks[i](b).flatten(2).transpose(1,2)
            v = self.Vs[i](b).flatten(2).transpose(1,2)
            y, _ = self.attn[i](q, k, v, need_weights=False)
            y = self.out[i](y.transpose(1,2).reshape(Bn,C,H,W))
            outs.append(a + y)
        return outs

def build_fusion(name, chs):
    name = name.lower()
    if name == 'late_logits':  return FusionLateLogits()
    if name == 'concat1x1':    return FusionConcat1x1(chs)
    if name == 'weighted_sum': return FusionWeightedSum(chs)
    if name == 'gated':        return FusionGated(chs)
    if name == 'film':         return FusionFiLM(chs)
    if name == 'cross_attn':   return FusionCrossAttention(chs, num_heads=4)
    raise ValueError(f'Unknown fusion: {name}')

# --- Main model ---
class DualSwinFusionSeg(nn.Module):
    def __init__(self, encoder_name, pretrained, img_size, fpn_channels, fusion_name, decoder_name='unetplusplus'):
        super().__init__()
        self.enc_rgb = make_swin_features(encoder_name, pretrained=pretrained, img_size=img_size)
        self.enc_aux = make_swin_features(encoder_name, pretrained=pretrained, img_size=img_size)
        adapt_patch_embed_in_chans(self.enc_aux, 4)
        self.chs = self.enc_rgb.feature_info.channels()
        self.fusion  = build_fusion(fusion_name, self.chs)
        self.decoder = build_decoder(decoder_name, self.chs, fpn_channels=fpn_channels)
        self.head    = nn.Conv2d(fpn_channels, 1, kernel_size=1)
        self.img_size = img_size
        self.fusion_name = fusion_name

    def _encode_rgb(self, x): return to_nchw(self.enc_rgb(x), self.chs)
    def _encode_aux(self, x): return to_nchw(self.enc_aux(x), self.chs)
    def _decode(self, feats):
        x = self.decoder(feats)
        return F.interpolate(self.head(x), size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)

    def forward(self, rgb, aux4):
        fr = self._encode_rgb(rgb)
        fa = self._encode_aux(aux4)
        if isinstance(self.fusion, FusionLateLogits):
            return 0.5 * (self._decode(fr) + self._decode(fa))
        return self._decode(self.fusion(fr, fa))

print('Model classes defined.')

Model classes defined.


In [28]:
# ============================================================
# Cell 5 — ★ v4 InferenceDataset (per-image norm) + load_fold_models
# ============================================================
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class InferenceDataset(Dataset):
    """Load test TIFs with v4 per-image percentile normalization."""
    def __init__(self, image_dir, img_size, band_indices, means, stds,
                 rgb_stat_idx, aux_stat_idx):
        self.paths = sorted(Path(image_dir).glob("*.tif")) + sorted(Path(image_dir).glob("*.TIF"))
        self.img_size = img_size
        self.band_indices = band_indices
        self.means = np.array(means, dtype=np.float32)
        self.stds  = np.array(stds,  dtype=np.float32)
        self.rgb_idx = rgb_stat_idx
        self.aux_idx = aux_stat_idx
        if len(self.paths) == 0:
            self.paths = sorted(Path(image_dir).glob("*.npy"))
        print(f"[InferenceDataset] Found {len(self.paths)} files in {image_dir}")

    def __len__(self): return len(self.paths)

    def _standardize(self, arr_chw, stat_idx):
        out = arr_chw.copy()
        for i, si in enumerate(stat_idx):
            out[i] = (out[i] - self.means[si]) / (self.stds[si] + 1e-6)
        return out

    def __getitem__(self, idx):
        p = self.paths[idx]
        name = p.name
        try:
            import rasterio as rio
            with rio.open(str(p)) as src:
                arr = src.read(self.band_indices).astype(np.float32)
        except Exception:
            arr = np.load(str(p)).astype(np.float32)
            if arr.ndim == 2:
                arr = arr[None]

        # ★ v4: Per-image percentile normalization (domain-invariant)
        arr = normalize_bands_per_image(arr)

        _, H, W = arr.shape
        if H != self.img_size or W != self.img_size:
            arr = np.stack([
                cv2.resize(arr[c], (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
                for c in range(arr.shape[0])
            ], axis=0)

        rgb_arr = arr[0:3]
        aux_arr = arr[3:7]
        rgb_arr = self._standardize(rgb_arr, self.rgb_idx)
        aux_arr = self._standardize(aux_arr, self.aux_idx)
        return torch.from_numpy(rgb_arr), torch.from_numpy(aux_arr), name


def load_fold_models(ckpt_dir, num_folds, encoder_name, img_size, fpn_channels, fusion_name,
                     decoder_name, device):
    """Load all fold checkpoints and return list of eval-mode models."""
    models = []
    ckpt_dir = Path(ckpt_dir)
    for fold in range(1, num_folds + 1):
        ckpt_path = ckpt_dir / f"fold{fold}_best.pt"
        if not ckpt_path.exists():
            print(f"  [WARNING] Checkpoint not found: {ckpt_path} — skipping fold {fold}")
            continue
        model = DualSwinFusionSeg(
            encoder_name=encoder_name,
            pretrained=False,
            img_size=img_size,
            fpn_channels=fpn_channels,
            fusion_name=fusion_name,
            decoder_name=decoder_name,
        ).to(device)
        ckpt = torch.load(str(ckpt_path), map_location=device)
        state = ckpt.get('model', ckpt.get('model_state', ckpt.get('state_dict', ckpt)))
        missing, unexpected = model.load_state_dict(state, strict=True)
        if missing:    print(f"  fold{fold} missing keys: {missing[:5]}")
        if unexpected: print(f"  fold{fold} unexpected keys: {unexpected[:5]}")
        model.eval()
        best_metrics = ckpt.get('best_metrics', {})
        val_iou = best_metrics.get('miou', best_metrics.get('val_miou',
                  ckpt.get('val_miou', ckpt.get('best_miou', None))))
        print(f"  fold{fold} loaded — val_mIoU: {val_iou:.4f}" if val_iou is not None else f"  fold{fold} loaded")
        models.append(model)
    print(f"\n[load_fold_models] {len(models)} / {num_folds} folds loaded.")
    return models

# ---- Load everything ----
# ★ v4: No STATS_LOW / STATS_HIGH needed — per-image normalization is self-contained
test_dataset = InferenceDataset(
    image_dir   = TEST_DATA_DIR,
    img_size    = IMG_SIZE,
    band_indices= BAND_INDICES,
    means       = MEANS,
    stds        = STDS,
    rgb_stat_idx= RGB_STAT_IDX,
    aux_stat_idx= AUX_STAT_IDX,
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
)

fold_models = load_fold_models(
    ckpt_dir     = CHECKPOINT_DIR,
    num_folds    = NUM_FOLDS,
    encoder_name = ENCODER_NAME,
    img_size     = IMG_SIZE,
    fpn_channels = FPN_CHANNELS,
    fusion_name  = FUSION_NAME,
    decoder_name = DECODER_NAME,
    device       = DEVICE,
)

print(f"\nTest samples: {len(test_dataset)} | Batches: {len(test_loader)}")

[InferenceDataset] Found 276 files in data/phase2_dataset/test/images


  fold1 loaded
  fold2 loaded
  fold3 loaded
  fold4 loaded
  fold5 loaded

[load_fold_models] 5 / 5 folds loaded.

Test samples: 276 | Batches: 35


In [29]:

# ============================================================
# Cell 6 — TTA + Ensemble Inference → Submission ZIP
# ============================================================
import zipfile
import rasterio

def _write_mask_tiff(mask01, out_path, height=128, width=128):
    """Write binary mask TIF, resizing with INTER_NEAREST if needed."""
    if mask01.shape[0] != height or mask01.shape[1] != width:
        mask01 = cv2.resize(
            mask01.astype(np.uint8), (width, height),
            interpolation=cv2.INTER_NEAREST,
        )
    with rasterio.open(
        out_path, 'w', driver='GTiff',
        height=height, width=width,
        count=1, dtype=rasterio.uint8,
    ) as dst:
        dst.write(mask01.astype(np.uint8), 1)


@torch.no_grad()
def tta_predict(model, rgb, aux):
    """4-fold TTA: original + hflip + vflip + rot90."""
    def infer(r, a): return torch.sigmoid(model(r, a))
    p  = infer(rgb, aux)
    p += infer(torch.flip(rgb, [-1]), torch.flip(aux, [-1])).flip(-1)                          # hflip
    p += infer(torch.flip(rgb, [-2]), torch.flip(aux, [-2])).flip(-2)                          # vflip
    p += infer(torch.rot90(rgb, 1, [-2, -1]), torch.rot90(aux, 1, [-2, -1])).rot90(3, [-2, -1])  # rot90
    return p / 4.0


@torch.no_grad()
def ensemble_predict(models, loader, out_dir, thresh=0.5, orig_size=128, use_tta=True):
    """
    Average probabilities across all fold models (at IMG_SIZE),
    threshold, then write masks at orig_size via _write_mask_tiff.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    written = 0

    for batch in tqdm(loader, desc="Inference"):
        rgb, aux, names = batch
        rgb = rgb.to(DEVICE, non_blocking=True)
        aux = aux.to(DEVICE, non_blocking=True)

        # Average probability across all fold models (at model resolution = IMG_SIZE)
        prob = torch.zeros(rgb.shape[0], 1, IMG_SIZE, IMG_SIZE, device=DEVICE)
        for model in models:
            if use_tta:
                prob += tta_predict(model, rgb, aux)
            else:
                prob += torch.sigmoid(model(rgb, aux))
        prob /= len(models)

        # Threshold at model resolution → binary masks (B, H, W) uint8
        masks = (prob[:, 0] > thresh).cpu().numpy().astype(np.uint8)

        # Write each mask as a TIF at orig_size (INTER_NEAREST resize inside _write_mask_tiff)
        for mask, name in zip(masks, names):
            out_path = str(out_dir / name)
            _write_mask_tiff(mask, out_path, height=orig_size, width=orig_size)
            written += 1

    print(f"[ensemble_predict] Wrote {written} mask TIFs → {out_dir}")
    return out_dir


def zip_predictions(pred_dir, zip_path):
    """Zip all predicted TIFs into a single submission archive."""
    pred_dir = Path(pred_dir)
    tifs = sorted(pred_dir.glob("*.tif")) + sorted(pred_dir.glob("*.TIF"))
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for t in tifs:
            zf.write(t, arcname=t.name)
    print(f"[zip_predictions] {len(tifs)} TIFs → {zip_path}")


# ---- Run inference ----
pred_dir = Path(OUTPUT_DIR) / "predictions"
zip_path = str(Path(OUTPUT_DIR) / "dual_swin_kfold_domain_adapted.zip")

ensemble_predict(
    models    = fold_models,
    loader    = test_loader,
    out_dir   = str(pred_dir),
    thresh    = THRESH,
    orig_size = ORIG_SIZE,
    use_tta   = USE_TTA,
)

zip_predictions(pred_dir, zip_path)
print(f"\nSubmission ready: {zip_path}")


Inference: 100%|██████████| 35/35 [00:15<00:00,  2.24it/s]

[ensemble_predict] Wrote 276 mask TIFs → output/nb_inference_output/predictions
[zip_predictions] 276 TIFs → output/nb_inference_output/dual_swin_kfold_domain_adapted.zip

Submission ready: output/nb_inference_output/dual_swin_kfold_domain_adapted.zip


In [30]:
# # ============================================================
# # Cell 7 — Optional: Compare predictions vs reference (Phase 1 sanity check)
# # Set REFERENCE_SUBMISSION_DIR to a folder of ground-truth / reference masks.
# # ============================================================
# if REFERENCE_SUBMISSION_DIR is not None:
#     ref_dir  = Path(REFERENCE_SUBMISSION_DIR)
#     pred_dir = Path(OUTPUT_DIR) / "predictions"
#     ref_files  = sorted(ref_dir.glob("*.tif"))
#     ious = []
#     mismatches = []
#     for ref_path in tqdm(ref_files, desc="Comparing"):
#         pred_path = pred_dir / ref_path.name
#         if not pred_path.exists():
#             mismatches.append(ref_path.name)
#             continue
#         with rio.open(str(ref_path))  as rf: ref_mask  = (rf.read(1) > 0).astype(np.uint8)
#         with rio.open(str(pred_path)) as pf: pred_mask = (pf.read(1) > 0).astype(np.uint8)
#         # Resize pred to match ref if needed
#         if ref_mask.shape != pred_mask.shape:
#             pred_mask = cv2.resize(pred_mask, (ref_mask.shape[1], ref_mask.shape[0]),
#                                    interpolation=cv2.INTER_NEAREST)
#         inter = int((ref_mask & pred_mask).sum())
#         union = int((ref_mask | pred_mask).sum())
#         iou   = inter / (union + 1e-6)
#         ious.append(iou)

#     mean_iou = float(np.mean(ious)) if ious else 0.0
#     print(f"Reference comparison over {len(ious)} tiles")
#     print(f"  Mean IoU  : {mean_iou:.4f}")
#     print(f"  Min / Max : {min(ious):.4f} / {max(ious):.4f}")
#     if mismatches:
#         print(f"  Missing in predictions ({len(mismatches)}): {mismatches[:10]}")
# else:
#     print("REFERENCE_SUBMISSION_DIR is None — skipping comparison.")

## v4 Domain Adaptation — Per-Image Normalization

| | |
|---|---|
| **Model** | `DualSwinFusionSeg` — UNet++ decoder, concat1x1 fusion |
| **Encoder** | `swinv2_small_window8_256` (dual: RGB 3ch + AUX 4ch) |
| **Normalization** | ★ Per-image percentile (P1/P99 per band per image) → [0,1] → z-score |
| **Stats** | Loaded from `norm_stats_v4.json` (z-score mean/std only) |
| **Ensemble** | 5 folds × 4-fold TTA |
| **Output** | `v4_inference_output/submission_v4.zip` |

**Why per-image normalization?**
- DEM shift: Train [-1345, 3110] vs Test [4275, 7232] → zero overlap
- Global percentile clipping makes test DEM a constant 1.0
- Per-image P1/P99 captures relative terrain shape regardless of absolute elevation

In [35]:
# ============================================================
# Cell 8 — Pixel-level comparison vs reference ZIP
# ============================================================
import zipfile, io
import numpy as np
import rasterio

REF_ZIP  = "output/inference_output_phase2/submission.zip"
PRED_ZIP = "output/nb_inference_output/dual_swin_kfold_domain_adapted.zip"

def load_masks_from_zip(zip_path):
    """Returns dict {filename: binary_mask_ndarray}."""
    masks = {}
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if not name.lower().endswith(".tif"):
                continue
            data = zf.read(name)
            with rasterio.open(io.BytesIO(data)) as src:
                m = (src.read(1) > 0).astype(np.uint8)
            masks[name] = m
    return masks

print("Loading reference …")
ref_masks  = load_masks_from_zip(REF_ZIP)
print("Loading predictions …")
pred_masks = load_masks_from_zip(PRED_ZIP)

common = sorted(set(ref_masks) & set(pred_masks))
only_ref  = set(ref_masks)  - set(pred_masks)
only_pred = set(pred_masks) - set(ref_masks)

print(f"\nFiles in reference  : {len(ref_masks)}")
print(f"Files in prediction : {len(pred_masks)}")
print(f"Common files        : {len(common)}")
if only_ref:  print(f"  Only in ref  : {sorted(only_ref)[:10]}")
if only_pred: print(f"  Only in pred : {sorted(only_pred)[:10]}")

total_pixels  = 0
total_changed = 0
changed_files = []

for name in common:
    r = ref_masks[name]
    p = pred_masks[name]
    if r.shape != p.shape:
        import cv2
        p = cv2.resize(p, (r.shape[1], r.shape[0]), interpolation=cv2.INTER_NEAREST)
    diff = int((r != p).sum())
    total_pixels  += r.size
    total_changed += diff
    if diff > 0:
        changed_files.append((name, diff, r.size))

changed_files.sort(key=lambda x: -x[1])

print(f"\n{'='*55}")
print(f"Total pixels compared : {total_pixels:,}")
print(f"Changed pixels        : {total_changed:,}  ({100*total_changed/max(total_pixels,1):.4f} %)")
print(f"Files with changes    : {len(changed_files)} / {len(common)}")
print(f"{'='*55}")
if changed_files:
    print("\nTop-10 files by pixel diff:")
    print(f"  {'File':<45} {'Changed':>8}  {'%':>7}")
    for fname, diff, size in changed_files[:10]:
        print(f"  {fname:<45} {diff:>8,}  {100*diff/size:>6.2f}%")


Loading reference …
Loading predictions …

Files in reference  : 276
Files in prediction : 276
Common files        : 276

Total pixels compared : 4,521,984
Changed pixels        : 75,222  (1.6635 %)
Files with changes    : 276 / 276

Top-10 files by pixel diff:
  File                                           Changed        %
  col00021_row00011.tif                            1,439    8.78%
  col00013_row00008.tif                            1,415    8.64%
  col00027_row00006.tif                            1,357    8.28%
  col00024_row00008.tif                            1,203    7.34%
  col00024_row00018.tif                            1,175    7.17%
  col00024_row00007.tif                            1,016    6.20%
  col00020_row00014.tif                              980    5.98%
  col00006_row00005.tif                              964    5.88%
  col00020_row00007.tif                              964    5.88%
  col00010_row00009.tif                              955    5.83%
